# Cross-Basin Tropical Cyclone Transfer Learning — Phase 1: Naive Baseline

**Goal:** Train a CNN+MLP model on Western Pacific (WP) data and evaluate zero-shot transfer to other basins (SP, NA, EP, NI, SI) for:
- **Direction prediction** (8-class: E, SE, S, SW, W, NW, N, NE)
- **Intensity change prediction** (4-class)

This establishes the baseline transfer gap that later phases (domain adaptation, physics-informed features) will aim to close.

## Section 0: Setup & Imports

In [ ]:
import subprocess, sys
for pkg in ["torch", "torchvision", "scikit-learn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from pathlib import Path
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, accuracy_score
from collections import Counter

# ── Constants ──
TCND_ROOT = Path("data/tropicyclonenet/TCND_test/TCND_test")
BASINS = ["WP", "EP", "NA", "NI", "SI", "SP"]
DIR_LABELS = ["E", "SE", "S", "SW", "W", "NW", "N", "NE"]
INTE_LABELS = ["Weakening", "Steady", "Slow-intens.", "Rapid-intens."]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Data root exists: {TCND_ROOT.exists()}")

## Section 1: Data Indexing

Walk the Env-Data directories to build a master index of all samples with their labels and file paths.

In [ ]:
def build_master_index(tcnd_root):
    """Build DataFrame indexing every sample across all basins."""
    rows = []
    env_root = tcnd_root / "Env-Data"
    nc_root = tcnd_root / "Data3D"

    for basin in BASINS:
        basin_dir = env_root / basin
        if not basin_dir.exists():
            continue
        for year_dir in sorted(basin_dir.iterdir()):
            if not year_dir.is_dir():
                continue
            year = year_dir.name
            for storm_dir in sorted(year_dir.iterdir()):
                if not storm_dir.is_dir():
                    continue
                storm = storm_dir.name
                for npy_file in sorted(storm_dir.glob("*.npy")):
                    dt = npy_file.stem  # e.g. "2017081018"
                    nc_file = nc_root / basin / year / storm / f"TCND_{storm}_{dt}_sst_z_u_v.nc"
                    # Load labels without loading full data
                    env = np.load(npy_file, allow_pickle=True).item()
                    rows.append({
                        "basin": basin,
                        "year": year,
                        "storm": storm,
                        "datetime": dt,
                        "env_path": str(npy_file),
                        "nc_path": str(nc_file),
                        "direction": int(env["future_direction24"]),
                        "intensity": int(env["future_inte_change24"]),
                    })

    df = pd.DataFrame(rows)
    return df

master_index = build_master_index(TCND_ROOT)
print(f"Total samples: {len(master_index)}")
print(f"\nSamples with valid direction label (≠ -1):")
valid_dir = master_index[master_index["direction"] != -1]
print(valid_dir.groupby("basin").size().to_string())
print(f"\nSamples with valid intensity label (≠ -1):")
valid_int = master_index[master_index["intensity"] != -1]
print(valid_int.groupby("basin").size().to_string())

## Section 2: Env-Data Feature Engineering

Build a 94-dimensional feature vector from each Env-Data dict:
- wind (1) + move_velocity (1) + intensity_class (6) + month (12) + area (6) + location_long (36) + location_lat (12) + history_direction12 (8) + history_direction24 (8) + history_inte_change24 (4) = 94

In [ ]:
def build_env_vector(env_dict):
    """Extract 94-dim feature vector from Env-Data dict.
    
    Handles type inconsistency: history fields are int -1 when missing,
    ndarray when present.
    """
    parts = []

    # Scalar features (cast to float)
    parts.append(np.array([float(env_dict["wind"])]))                # 1
    parts.append(np.array([float(env_dict["move_velocity"])]))       # 1

    # One-hot / categorical arrays
    parts.append(np.asarray(env_dict["intensity_class"], dtype=np.float32))  # 6
    parts.append(np.asarray(env_dict["month"], dtype=np.float32))            # 12
    parts.append(np.asarray(env_dict["area"], dtype=np.float32))             # 6
    parts.append(np.asarray(env_dict["location_long"], dtype=np.float32))    # 36
    parts.append(np.asarray(env_dict["location_lat"], dtype=np.float32))     # 12

    # History fields: int -1 when missing, ndarray when present
    for key, dim in [("history_direction12", 8),
                     ("history_direction24", 8),
                     ("history_inte_change24", 4)]:
        val = env_dict[key]
        if isinstance(val, np.ndarray):
            parts.append(val.astype(np.float32))
        else:
            parts.append(np.zeros(dim, dtype=np.float32))

    vec = np.concatenate(parts).astype(np.float32)
    assert vec.shape == (94,), f"Expected 94-dim, got {vec.shape}"
    return vec

# Quick test
sample_env = np.load(master_index.iloc[0]["env_path"], allow_pickle=True).item()
test_vec = build_env_vector(sample_env)
print(f"Env vector shape: {test_vec.shape}")
print(f"Sample values (first 10): {test_vec[:10]}")

## Section 3: PyTorch Dataset

Loads each sample's 13-channel grid (sst + u/v/z at 4 pressure levels) and 94-dim env vector.

In [ ]:
class TCNDDataset(Dataset):
    """PyTorch dataset for TCND samples.
    
    Each sample returns:
        grid:  (13, 81, 81) tensor — sst(1) + u(4) + v(4) + z(4)
        env:   (94,) tensor — tabular features
        label: int — class index
    """

    def __init__(self, index_df, task="direction"):
        assert task in ("direction", "intensity")
        self.task = task
        # Filter to valid labels
        label_col = "direction" if task == "direction" else "intensity"
        self.df = index_df[index_df[label_col] != -1].reset_index(drop=True)
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load env vector
        env_dict = np.load(row["env_path"], allow_pickle=True).item()
        env_vec = build_env_vector(env_dict)

        # Load 3D grid: stack into 13 channels
        with h5py.File(row["nc_path"], "r") as f:
            sst = f["sst"][:]                    # (81, 81)
            u = f["u"][0]                        # (4, 81, 81)
            v = f["v"][0]                        # (4, 81, 81)
            z = f["z"][0]                        # (4, 81, 81)

        # Replace NaNs with 0
        sst = np.nan_to_num(sst, nan=0.0)
        grid = np.concatenate([sst[np.newaxis], u, v, z], axis=0)  # (13, 81, 81)

        label = int(row[self.label_col])

        return (
            torch.tensor(grid, dtype=torch.float32),
            torch.tensor(env_vec, dtype=torch.float32),
            label,
        )

# Quick test
test_ds = TCNDDataset(master_index[master_index["basin"] == "WP"], task="direction")
g, e, l = test_ds[0]
print(f"Grid shape: {g.shape}, Env shape: {e.shape}, Label: {l}")
print(f"WP direction dataset size: {len(test_ds)}")

## Section 4: Train/Val Split

**Storm-level split** to avoid data leakage from consecutive 6-hour timesteps within the same storm.

In [ ]:
def storm_split(index_df, basin="WP", val_frac=0.2, seed=42):
    """Split by storm (not by sample) to prevent temporal leakage."""
    basin_df = index_df[index_df["basin"] == basin].copy()
    storms = basin_df["storm"].unique()
    rng = np.random.RandomState(seed)
    rng.shuffle(storms)

    n_val = max(1, int(len(storms) * val_frac))
    val_storms = set(storms[:n_val])
    train_storms = set(storms[n_val:])

    train_df = basin_df[basin_df["storm"].isin(train_storms)]
    val_df = basin_df[basin_df["storm"].isin(val_storms)]

    print(f"{basin} split: {len(train_storms)} train storms ({len(train_df)} samples), "
          f"{len(val_storms)} val storms ({len(val_df)} samples)")
    return train_df, val_df

wp_train_df, wp_val_df = storm_split(master_index, basin="WP")

In [ ]:
BATCH_SIZE = 32

def make_loaders(train_df, val_df, task="direction", batch_size=BATCH_SIZE):
    train_ds = TCNDDataset(train_df, task=task)
    val_ds = TCNDDataset(val_df, task=task)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=0, pin_memory=False)
    return train_loader, val_loader

dir_train_loader, dir_val_loader = make_loaders(wp_train_df, wp_val_df, task="direction")
print(f"Direction — Train batches: {len(dir_train_loader)}, Val batches: {len(dir_val_loader)}")

int_train_loader, int_val_loader = make_loaders(wp_train_df, wp_val_df, task="intensity")
print(f"Intensity — Train batches: {len(int_train_loader)}, Val batches: {len(int_val_loader)}")

## Section 5: Model Architecture — CycloneNet

A CNN+MLP hybrid:
- **CNN branch**: 3 conv layers on the 13-channel (81×81) grid → 128-dim
- **MLP branch**: 1 hidden layer on the 94-dim env vector → 64-dim
- **Fusion**: concatenate → 2 FC layers → class logits

~200K parameters, trainable on CPU in ~4 minutes for 15 epochs.

In [ ]:
class CycloneNet(nn.Module):
    """CNN + MLP hybrid for TC forecasting."""

    def __init__(self, num_classes=8, env_dim=94):
        super().__init__()

        # CNN branch: 13-channel (81x81) grid
        self.cnn = nn.Sequential(
            nn.Conv2d(13, 32, kernel_size=5, stride=2, padding=2),  # → 41x41
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # → 21x21
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # → 11x11
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                                # → 1x1
            nn.Flatten(),                                           # → 128
        )

        # MLP branch: 94-dim env features
        self.mlp = nn.Sequential(
            nn.Linear(env_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Fusion head
        self.head = nn.Sequential(
            nn.Linear(128 + 64, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, grid, env):
        cnn_feat = self.cnn(grid)       # (B, 128)
        mlp_feat = self.mlp(env)        # (B, 64)
        fused = torch.cat([cnn_feat, mlp_feat], dim=1)  # (B, 192)
        return self.head(fused)         # (B, num_classes)

# Parameter count
model_test = CycloneNet(num_classes=8)
n_params = sum(p.numel() for p in model_test.parameters())
print(f"CycloneNet parameters: {n_params:,}")
del model_test

## Section 6: Training Loop

CrossEntropyLoss with inverse-frequency class weights to handle class imbalance. Adam optimizer, 15 epochs.

In [ ]:
def compute_class_weights(dataset):
    """Inverse-frequency weights for CrossEntropyLoss."""
    labels = [dataset[i][2] for i in range(len(dataset))]
    counts = Counter(labels)
    n_classes = max(counts.keys()) + 1
    total = len(labels)
    weights = torch.zeros(n_classes)
    for c in range(n_classes):
        if counts[c] > 0:
            weights[c] = total / (n_classes * counts[c])
        else:
            weights[c] = 1.0
    return weights


def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3, task_name=""):
    """Train with class-weighted CE loss; return history dict."""
    model = model.to(DEVICE)

    # Compute class weights from training set
    weights = compute_class_weights(train_loader.dataset).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_state = None

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for grid, env, labels in train_loader:
            grid, env = grid.to(DEVICE), env.to(DEVICE)
            labels = torch.tensor(labels, dtype=torch.long).to(DEVICE) if not isinstance(labels, torch.Tensor) else labels.to(DEVICE)

            optimizer.zero_grad()
            logits = model(grid, env)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * grid.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += grid.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # ── Validate ──
        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for grid, env, labels in val_loader:
                grid, env = grid.to(DEVICE), env.to(DEVICE)
                labels = torch.tensor(labels, dtype=torch.long).to(DEVICE) if not isinstance(labels, torch.Tensor) else labels.to(DEVICE)
                logits = model(grid, env)
                loss = criterion(logits, labels)
                running_loss += loss.item() * grid.size(0)
                correct += (logits.argmax(1) == labels).sum().item()
                total += grid.size(0)

        val_loss = running_loss / total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f"  Epoch {epoch+1:2d}/{epochs}  "
              f"train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.3f}"
              f"{'  ★' if val_acc >= best_val_acc else ''}")

    # Restore best model
    model.load_state_dict(best_state)
    model = model.to(DEVICE)
    print(f"\n{task_name} best val accuracy: {best_val_acc:.3f}")
    return history


def plot_training_curves(history, title="Training Curves"):
    """Plot loss and accuracy curves side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    epochs = range(1, len(history["train_loss"]) + 1)

    ax1.plot(epochs, history["train_loss"], "o-", label="Train")
    ax1.plot(epochs, history["val_loss"], "s-", label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title(f"{title} — Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, history["train_acc"], "o-", label="Train")
    ax2.plot(epochs, history["val_acc"], "s-", label="Val")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title(f"{title} — Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Section 7: Train Direction Model (8-class)

In [ ]:
print("Training direction model (8-class) on WP...")
dir_model = CycloneNet(num_classes=8)
dir_history = train_model(dir_model, dir_train_loader, dir_val_loader,
                          epochs=15, lr=1e-3, task_name="Direction")

In [ ]:
plot_training_curves(dir_history, title="Direction (8-class)")

## Section 8: Train Intensity Model (4-class)

In [ ]:
print("Training intensity model (4-class) on WP...")
int_model = CycloneNet(num_classes=4)
int_history = train_model(int_model, int_train_loader, int_val_loader,
                          epochs=15, lr=1e-3, task_name="Intensity")

In [ ]:
plot_training_curves(int_history, title="Intensity Change (4-class)")

## Section 9: Cross-Basin Evaluation

Evaluate both models on all 6 basins to quantify the transfer gap.

In [ ]:
def evaluate_basin(model, basin_df, task="direction", batch_size=64):
    """Evaluate model on a basin's data. Returns accuracy, top-2 accuracy, and per-class accuracy."""
    ds = TCNDDataset(basin_df, task=task)
    if len(ds) == 0:
        return {"accuracy": float("nan"), "top2_accuracy": float("nan"),
                "per_class": {}, "n_samples": 0, "y_true": [], "y_pred": []}

    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    model.eval()

    all_preds, all_labels = [], []
    top2_correct = 0

    with torch.no_grad():
        for grid, env, labels in loader:
            grid, env = grid.to(DEVICE), env.to(DEVICE)
            logits = model(grid, env)
            preds = logits.argmax(1).cpu().numpy()
            top2 = logits.topk(2, dim=1).indices.cpu().numpy()

            labels_np = np.array(labels)
            all_preds.extend(preds)
            all_labels.extend(labels_np)
            top2_correct += sum(labels_np[i] in top2[i] for i in range(len(labels_np)))

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    top2_acc = top2_correct / len(all_labels)

    # Per-class accuracy
    n_classes = max(all_labels.max(), all_preds.max()) + 1
    per_class = {}
    for c in range(n_classes):
        mask = all_labels == c
        if mask.sum() > 0:
            per_class[c] = (all_preds[mask] == c).mean()

    return {
        "accuracy": acc,
        "top2_accuracy": top2_acc,
        "per_class": per_class,
        "n_samples": len(all_labels),
        "y_true": all_labels,
        "y_pred": all_preds,
    }

In [ ]:
# Evaluate direction and intensity models on all basins
results = []

for basin in BASINS:
    # Use val split for WP, full data for other basins
    if basin == "WP":
        basin_df = wp_val_df
        label = "WP (val)"
    else:
        basin_df = master_index[master_index["basin"] == basin]
        label = basin

    dir_res = evaluate_basin(dir_model, basin_df, task="direction")
    int_res = evaluate_basin(int_model, basin_df, task="intensity")

    results.append({
        "Basin": label,
        "N (dir)": dir_res["n_samples"],
        "Dir Acc": dir_res["accuracy"],
        "Dir Top-2": dir_res["top2_accuracy"],
        "N (int)": int_res["n_samples"],
        "Inte Acc": int_res["accuracy"],
        "_dir_res": dir_res,
        "_int_res": int_res,
    })

    print(f"{label:10s}  dir={dir_res['accuracy']:.3f} (top2={dir_res['top2_accuracy']:.3f})  "
          f"int={int_res['accuracy']:.3f}  n_dir={dir_res['n_samples']}  n_int={int_res['n_samples']}")

In [ ]:
# Build results table with transfer gaps
wp_dir_acc = results[0]["Dir Acc"]
wp_int_acc = results[0]["Inte Acc"]

table_rows = []
for r in results:
    table_rows.append({
        "Basin": r["Basin"],
        "N (dir)": r["N (dir)"],
        "Dir Acc": f"{r['Dir Acc']:.3f}",
        "Dir Top-2": f"{r['Dir Top-2']:.3f}",
        "N (int)": r["N (int)"],
        "Inte Acc": f"{r['Inte Acc']:.3f}",
        "Dir Gap": f"{r['Dir Acc'] - wp_dir_acc:+.3f}",
        "Inte Gap": f"{r['Inte Acc'] - wp_int_acc:+.3f}",
    })

results_table = pd.DataFrame(table_rows)
print("\n=== Cross-Basin Transfer Results ===")
print(results_table.to_string(index=False))

## Section 10: Confusion Matrices

Side-by-side heatmaps comparing WP (val) vs SP direction predictions. Row-normalized to show per-class recall — this reveals hemisphere bias (model predicts NW/W for SP when true direction is SE/SW).

In [ ]:
def plot_confusion_pair(res_a, res_b, label_a, label_b, class_names, title=""):
    """Side-by-side row-normalized confusion matrices."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for ax, res, label in [(ax1, res_a, label_a), (ax2, res_b, label_b)]:
        cm = confusion_matrix(res["y_true"], res["y_pred"],
                              labels=list(range(len(class_names))))
        # Row-normalize (recall per class)
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1  # avoid division by zero
        cm_norm = cm / row_sums

        im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(class_names)))
        ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels(class_names, rotation=45, ha="right")
        ax.set_yticklabels(class_names)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"{label} (acc={res['accuracy']:.3f})")

        # Annotate cells
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                val = cm_norm[i, j]
                color = "white" if val > 0.5 else "black"
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        color=color, fontsize=8)

    fig.colorbar(im, ax=[ax1, ax2], shrink=0.8, label="Recall")
    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("confusion_direction_wp_vs_sp.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: confusion_direction_wp_vs_sp.png")


# Find WP and SP results
wp_dir_res = results[0]["_dir_res"]
sp_dir_res = next(r["_dir_res"] for r in results if r["Basin"] == "SP")

plot_confusion_pair(wp_dir_res, sp_dir_res,
                    "WP (val)", "SP",
                    DIR_LABELS,
                    title="Direction Prediction: WP vs SP Confusion Matrices")

## Section 11: Transfer Gap Bar Chart

Grouped bar chart showing direction and intensity accuracy per basin, with WP reference line.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

basin_labels = [r["Basin"] for r in results]
dir_accs = [r["Dir Acc"] for r in results]
int_accs = [r["Inte Acc"] for r in results]

x = np.arange(len(basin_labels))
width = 0.35

bars1 = ax.bar(x - width/2, dir_accs, width, label="Direction (8-class)",
               color="#1b4f72", alpha=0.85)
bars2 = ax.bar(x + width/2, int_accs, width, label="Intensity (4-class)",
               color="#e67e22", alpha=0.85)

# Reference lines at WP val accuracy
ax.axhline(y=wp_dir_acc, color="#1b4f72", linestyle="--", alpha=0.5,
           label=f"WP dir ref ({wp_dir_acc:.3f})")
ax.axhline(y=wp_int_acc, color="#e67e22", linestyle="--", alpha=0.5,
           label=f"WP int ref ({wp_int_acc:.3f})")

ax.set_xlabel("Basin")
ax.set_ylabel("Accuracy")
ax.set_title("Cross-Basin Transfer: Direction & Intensity Accuracy")
ax.set_xticks(x)
ax.set_xticklabels(basin_labels)
ax.legend(loc="upper right")
ax.set_ylim(0, 1)
ax.grid(True, axis="y", alpha=0.3)

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("transfer_gap_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: transfer_gap_bar_chart.png")

## Section 12: Analysis & Next Steps

### Key Findings

**Direction prediction (8-class):**
- WP in-basin accuracy should be well above random (12.5%), demonstrating the model learns meaningful patterns
- SP transfer gap is expected to be the largest due to **hemisphere bias**: WP storms move predominantly NW/W while SP storms move SE/W/SW. The model will likely predict NW for SP storms when the true direction is SE.
- NA/EP (same hemisphere as WP) should transfer better than SI/SP (Southern Hemisphere)

**Intensity change prediction (4-class):**
- Intensity physics are more universal across basins (SST, shear drive intensification everywhere)
- Transfer gap should be smaller than direction, since intensity change is less hemisphere-dependent

### Why Does Transfer Fail?

The model has access to **basin-leaking features**:
- `area` (6-dim one-hot): directly encodes which basin the storm is from
- `location_long` (36-dim) and `location_lat` (12-dim): absolute position reveals hemisphere and basin
- These features let the model learn basin-specific shortcuts instead of universal physics

### Roadmap for Phase 2

1. **Remove basin-leaking features**: Drop `area`, `location_long`, `location_lat` from the env vector (reduces from 94 to 40 dims)
2. **Add physics-informed features**:
   - Vertical wind shear (computed from u/v at 200 and 850 hPa)
   - SST anomaly (deviation from tropical mean)
   - Maximum potential intensity (MPI)
3. **Hemisphere-aware augmentation**: Flip v-wind and mirror direction labels for Southern Hemisphere storms during training
4. **Domain-Adversarial Neural Network (DANN)**: Add gradient reversal layer to prevent the model from learning basin-discriminative features
5. **Multi-basin pretraining**: Train on all basins simultaneously, then fine-tune on target basin with few-shot learning